# ParlaMint-SI analysis dataset preparation
This notebook merges the initial ParlaSent-SI dataset, which contains sentence-level sentiment annotation scores, with aggregated utterance-level sentiment scores produced by a retrained Random Forest (RF) model to prepare a comprehensive dataset for analyzing sentiment trends at both:
- [sentence](Analysis_sentences.ipynb) level and 
- utterance levels, using both: 
    - [RF annotations](Analysis_utterances.ipynb) and
    - [best performing heuristic - characters-based average](./Analysis_heuristic.ipynb)) 

In [1]:
import pandas as pd
import numpy as np

## Processing ParlaSent-SI dataset with sentence level annotations

In [2]:
def process_chunk(df_chunk):
    print(df_chunk.head())
def process_jsonl(file_path, chunksize=1000):
    df_list=[]
    for df_chunk in pd.read_json(file_path, lines=True, chunksize=chunksize):
        df_list.append(df_chunk)
    
    full_df = pd.concat(df_list, ignore_index=True)
    return full_df

file_path = '../../../ParlaSent-SI/ParlaMint-SI.meta.jsonl'
df_sent = process_jsonl(file_path, chunksize=1000)
df_sent.head()

,sent_id,text,logit,newdoc id,text_en,metadata
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Spoštovane kolegice poslanke in kolegi poslanc...,4.052144,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,"Dear fellow Members and fellow Members, ladies...",{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Začenjam z nadaljevanjem 99. izredne seje Drža...,3.223034,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,I'm starting with the 99th Extraordinary Meeti...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,"Obveščen sem, da se današnje seje ne morejo ud...",1.778516,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,I am informed that today's meeting cannot be a...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Vse prisotne lepo pozdravljam!,4.814360,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,I say hello to everyone who's here!,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Prehajamo na 10.,3.219104,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,We're moving on to 10.,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...


In [3]:
order = ['ID', 'sent_id', 'text', 'text_en', 'metadata', 'sent_annotations']
df_sent = df_sent.rename(columns={'logit':'sent_annotations', 'newdoc id':'ID'})
df_sent = df_sent[order]
df_sent.head()

,ID,sent_id,text,text_en,metadata,sent_annotations
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Spoštovane kolegice poslanke in kolegi poslanc...,"Dear fellow Members and fellow Members, ladies...",{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,4.052144
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Začenjam z nadaljevanjem 99. izredne seje Drža...,I'm starting with the 99th Extraordinary Meeti...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,3.223034
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,"Obveščen sem, da se današnje seje ne morejo ud...",I am informed that today's meeting cannot be a...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,1.778516
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Vse prisotne lepo pozdravljam!,I say hello to everyone who's here!,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,4.814360
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Prehajamo na 10.,We're moving on to 10.,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,3.219104


## Mapping s-level annotations to labels (3- and 6-class schema)

In [4]:
label_mapping = {
    0: 'Negative',
    1: 'Mixed Negative',
    2: 'Neutral Negative',
    3: 'Neutral Positive',
    4: 'Mixed Positive',
    5: 'Positive'
}

annotations = np.clip(np.round(df_sent['sent_annotations']), 0, 5).astype(int)
df_sent['annotations_clip'] = annotations
df_sent['s_labels'] = df_sent['annotations_clip'].map(label_mapping)

In [5]:
mapping = {
    0: 'Negative',
    1: 'Neutral',   
    2: 'Positive',
}

sentiment = np.floor_divide(df_sent['annotations_clip'], 2)
df_sent['sentiment_no'] = sentiment
df_sent['s_sentiment'] = df_sent['sentiment_no'].map(mapping)
df_sent = df_sent.drop(columns=['annotations_clip', 'sentiment_no'])

df_sent.head()

,ID,sent_id,text,text_en,metadata,sent_annotations,s_labels,s_sentiment
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Spoštovane kolegice poslanke in kolegi poslanc...,"Dear fellow Members and fellow Members, ladies...",{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,4.052144,Mixed Positive,Positive
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Začenjam z nadaljevanjem 99. izredne seje Drža...,I'm starting with the 99th Extraordinary Meeti...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,3.223034,Neutral Positive,Neutral
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,"Obveščen sem, da se današnje seje ne morejo ud...",I am informed that today's meeting cannot be a...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,1.778516,Neutral Negative,Neutral
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Vse prisotne lepo pozdravljam!,I say hello to everyone who's here!,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,4.814360,Positive,Positive
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Prehajamo na 10.,We're moving on to 10.,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,3.219104,Neutral Positive,Neutral


## Merging utterance sentiment scores 
Adding utterance level scores -- specifically, Random Forest annotations and best-performing heuristic annotations (char_avg)

In [6]:
df_utt = pd.read_csv('../../Datasets/ParlaMint-SI/utt_labels_RF.tsv', delimiter='\t', encoding='utf-8')
df_char= pd.read_csv('../../Datasets/ParlaMint-SI/utt_labels_heuristic.tsv', delimiter='\t', encoding='utf-8')

In [7]:
df_utt = df_utt.rename(columns={'annotations':'u_annotations', 'labels':'u_label', 'sentiment': 'u_sentiment'})

In [8]:
df_utt.head()

,ID,u_annotations,u_label,u_sentiment
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,2.42,Neutral Negative,Neutral
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u2,3.88,Mixed Positive,Positive
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u3,3.00,Neutral Positive,Neutral
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u4,3.19,Neutral Positive,Neutral
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u5,3.00,Neutral Positive,Neutral


In [9]:
df_char = df_char.rename(columns={'annotations':'h_annotations', 'labels':'h_labels', 'sentiment':'h_sentiment'})
df_char.head()

,ID,h_annotations,h_labels,h_sentiment
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,2.401479,Neutral Negative,Neutral
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u2,2.805297,Neutral Positive,Neutral
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u3,3.365201,Neutral Positive,Neutral
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u4,3.148236,Neutral Positive,Neutral
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u5,3.271888,Neutral Positive,Neutral


### Check for no. of utterances

In [10]:
df1 = set(df_sent['ID'])
df2 = set(df_utt['ID'])
df3 = set(df_char['ID'])

count_df1 = len(df1)
count_df2 = len(df2)
count_df3 = len(df3)

print(f"Number of unique utt_id in df1: {count_df1}")
print(f"Number of unique ID in df2: {count_df2}")
print(f"Number of unique ID in df2: {count_df3}")

Number of unique utt_id in df1: 311347
Number of unique ID in df2: 311347
Number of unique ID in df2: 311347


### Merging the datasets

In [11]:
temp_dataset = pd.merge(df_sent, df_utt, on='ID')
temp_dataset

,ID,sent_id,text,text_en,metadata,sent_annotations,s_labels,s_sentiment,u_annotations,u_label,u_sentiment
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Spoštovane kolegice poslanke in kolegi poslanc...,"Dear fellow Members and fellow Members, ladies...",{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,4.052144,Mixed Positive,Positive,2.42,Neutral Negative,Neutral
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Začenjam z nadaljevanjem 99. izredne seje Drža...,I'm starting with the 99th Extraordinary Meeti...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,3.223034,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,"Obveščen sem, da se današnje seje ne morejo ud...",I am informed that today's meeting cannot be a...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,1.778516,Neutral Negative,Neutral,2.42,Neutral Negative,Neutral
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Vse prisotne lepo pozdravljam!,I say hello to everyone who's here!,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,4.814360,Positive,Positive,2.42,Neutral Negative,Neutral
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Prehajamo na 10.,We're moving on to 10.,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,3.219104,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral
...,...,...,...,...,...,...,...,...,...,...,...
3876178,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u246,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.seg5...,S tem je bil seznam vseh poslanskih vprašanj i...,"With this, the list of all the questions sent ...",{'Text_ID': 'ParlaMint-SI-en_2005-01-24-SDZ4-R...,2.761963,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral
3876179,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u246,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.seg5...,Zahvaljujem se gospodu predsedniku vlade in go...,I thank Mr Prime Minister and Mr Ministers for...,{'Text_ID': 'ParlaMint-SI-en_2005-01-24-SDZ4-R...,4.138685,Mixed Positive,Positive,2.42,Neutral Negative,Neutral
3876180,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u246,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.seg5...,S tem zaključujem 1. točko dnevnega reda.,I hereby conclude the first item of the agenda.,{'Text_ID': 'ParlaMint-SI-en_2005-01-24-SDZ4-R...,3.587247,Mixed Positive,Positive,2.42,Neutral Negative,Neutral
3876181,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u246,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.seg5...,Prekinjam 2. sejo Državnega zbora Republike Sl...,I am adjourning the 2nd session of the Nationa...,{'Text_ID': 'ParlaMint-SI-en_2005-01-24-SDZ4-R...,2.921267,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral


In [12]:
dataset = pd.merge(temp_dataset, df_char, on='ID')
dataset

,ID,sent_id,text,text_en,metadata,sent_annotations,s_labels,s_sentiment,u_annotations,u_label,u_sentiment,h_annotations,h_labels,h_sentiment
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Spoštovane kolegice poslanke in kolegi poslanc...,"Dear fellow Members and fellow Members, ladies...",{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,4.052144,Mixed Positive,Positive,2.42,Neutral Negative,Neutral,2.401479,Neutral Negative,Neutral
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Začenjam z nadaljevanjem 99. izredne seje Drža...,I'm starting with the 99th Extraordinary Meeti...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,3.223034,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral,2.401479,Neutral Negative,Neutral
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,"Obveščen sem, da se današnje seje ne morejo ud...",I am informed that today's meeting cannot be a...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,1.778516,Neutral Negative,Neutral,2.42,Neutral Negative,Neutral,2.401479,Neutral Negative,Neutral
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Vse prisotne lepo pozdravljam!,I say hello to everyone who's here!,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,4.814360,Positive,Positive,2.42,Neutral Negative,Neutral,2.401479,Neutral Negative,Neutral
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Prehajamo na 10.,We're moving on to 10.,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...,3.219104,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral,2.401479,Neutral Negative,Neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3876178,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u246,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.seg5...,S tem je bil seznam vseh poslanskih vprašanj i...,"With this, the list of all the questions sent ...",{'Text_ID': 'ParlaMint-SI-en_2005-01-24-SDZ4-R...,2.761963,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral,3.254007,Neutral Positive,Neutral
3876179,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u246,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.seg5...,Zahvaljujem se gospodu predsedniku vlade in go...,I thank Mr Prime Minister and Mr Ministers for...,{'Text_ID': 'ParlaMint-SI-en_2005-01-24-SDZ4-R...,4.138685,Mixed Positive,Positive,2.42,Neutral Negative,Neutral,3.254007,Neutral Positive,Neutral
3876180,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u246,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.seg5...,S tem zaključujem 1. točko dnevnega reda.,I hereby conclude the first item of the agenda.,{'Text_ID': 'ParlaMint-SI-en_2005-01-24-SDZ4-R...,3.587247,Mixed Positive,Positive,2.42,Neutral Negative,Neutral,3.254007,Neutral Positive,Neutral
3876181,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u246,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.seg5...,Prekinjam 2. sejo Državnega zbora Republike Sl...,I am adjourning the 2nd session of the Nationa...,{'Text_ID': 'ParlaMint-SI-en_2005-01-24-SDZ4-R...,2.921267,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral,3.254007,Neutral Positive,Neutral


## Adding metadata as columns

In [13]:
import ast

metadata_df = pd.json_normalize(dataset['metadata'])
dataset = pd.concat([dataset, metadata_df], axis=1).drop(columns=['metadata'])
dataset.head()


,ID,sent_id,text,text_en,sent_annotations,s_labels,s_sentiment,u_annotations,u_label,u_sentiment,...,Speaker_MP,Speaker_minister,Speaker_party,Speaker_party_name,Party_status,Party_orientation,Speaker_ID,Speaker_name,Speaker_gender,Speaker_birth
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Spoštovane kolegice poslanke in kolegi poslanc...,"Dear fellow Members and fellow Members, ladies...",4.052144,Mixed Positive,Positive,2.42,Neutral Negative,Neutral,...,MP,notMinister,DeSUS,Demokratična stranka upokojencev Slovenije,Coalition,Centre to centre-left,SimonovičBranko,"Simonovič, Branko",M,1953
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Začenjam z nadaljevanjem 99. izredne seje Drža...,I'm starting with the 99th Extraordinary Meeti...,3.223034,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral,...,MP,notMinister,DeSUS,Demokratična stranka upokojencev Slovenije,Coalition,Centre to centre-left,SimonovičBranko,"Simonovič, Branko",M,1953
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,"Obveščen sem, da se današnje seje ne morejo ud...",I am informed that today's meeting cannot be a...,1.778516,Neutral Negative,Neutral,2.42,Neutral Negative,Neutral,...,MP,notMinister,DeSUS,Demokratična stranka upokojencev Slovenije,Coalition,Centre to centre-left,SimonovičBranko,"Simonovič, Branko",M,1953
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Vse prisotne lepo pozdravljam!,I say hello to everyone who's here!,4.814360,Positive,Positive,2.42,Neutral Negative,Neutral,...,MP,notMinister,DeSUS,Demokratična stranka upokojencev Slovenije,Coalition,Centre to centre-left,SimonovičBranko,"Simonovič, Branko",M,1953
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Prehajamo na 10.,We're moving on to 10.,3.219104,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral,...,MP,notMinister,DeSUS,Demokratična stranka upokojencev Slovenije,Coalition,Centre to centre-left,SimonovičBranko,"Simonovič, Branko",M,1953


In [15]:
dataset = dataset.rename(columns={"sent_annotations":"s_annotations"})
dataset.head()

,ID,sent_id,text,text_en,s_annotations,s_labels,s_sentiment,u_annotations,u_label,u_sentiment,...,Speaker_MP,Speaker_minister,Speaker_party,Speaker_party_name,Party_status,Party_orientation,Speaker_ID,Speaker_name,Speaker_gender,Speaker_birth
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Spoštovane kolegice poslanke in kolegi poslanc...,"Dear fellow Members and fellow Members, ladies...",4.052144,Mixed Positive,Positive,2.42,Neutral Negative,Neutral,...,MP,notMinister,DeSUS,Demokratična stranka upokojencev Slovenije,Coalition,Centre to centre-left,SimonovičBranko,"Simonovič, Branko",M,1953
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Začenjam z nadaljevanjem 99. izredne seje Drža...,I'm starting with the 99th Extraordinary Meeti...,3.223034,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral,...,MP,notMinister,DeSUS,Demokratična stranka upokojencev Slovenije,Coalition,Centre to centre-left,SimonovičBranko,"Simonovič, Branko",M,1953
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,"Obveščen sem, da se današnje seje ne morejo ud...",I am informed that today's meeting cannot be a...,1.778516,Neutral Negative,Neutral,2.42,Neutral Negative,Neutral,...,MP,notMinister,DeSUS,Demokratična stranka upokojencev Slovenije,Coalition,Centre to centre-left,SimonovičBranko,"Simonovič, Branko",M,1953
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Vse prisotne lepo pozdravljam!,I say hello to everyone who's here!,4.814360,Positive,Positive,2.42,Neutral Negative,Neutral,...,MP,notMinister,DeSUS,Demokratična stranka upokojencev Slovenije,Coalition,Centre to centre-left,SimonovičBranko,"Simonovič, Branko",M,1953
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Prehajamo na 10.,We're moving on to 10.,3.219104,Neutral Positive,Neutral,2.42,Neutral Negative,Neutral,...,MP,notMinister,DeSUS,Demokratična stranka upokojencev Slovenije,Coalition,Centre to centre-left,SimonovičBranko,"Simonovič, Branko",M,1953


In [ ]:
dataset.to_csv('../../Datasets/ParlaMint-SI.tsv', sep='\t', encoding='utf-8', index=False)